# Stage 3 — Exploratory Data Analysis
### Walmart M5, store `CA_1`, department `FOODS_3` — Phase 1 (synopsis prep)

Loads the real, already-computed artifacts from Stages 1-2
(`data/interim/panel_ca1_foods3.parquet`,
`data/processed/demand_classification.parquet`,
`data/processed/selected_items.parquet`,
`data/processed/aggregate_ca1_foods3.parquet`) and works through the
Phase 1 EDA checklist. All computation logic lives in
`src/analysis/eda.py` (and `src/data/validator.py` for the continuity
checks, reused rather than duplicated) — cells here call those
functions and render the result.

Matplotlib is used throughout for consistency with
`notebooks/01_data_understanding.ipynb` and
`notebooks/03_demand_classification.ipynb`; there's no interactive-only
analysis here that would justify switching to Plotly for one section.

**No forecasting model, model fit, or forecast result appears in this
notebook.** Run `python -m src.data.preprocessor` and
`python -m src.analysis.demand_classification` first if the files
below don't exist yet.

In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing config.yaml is
    found. Jupyter's kernel working directory isn't consistent across
    setups (often the notebook's own folder, sometimes the launch
    directory), so this avoids assuming which one it is.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find config.yaml in any parent directory of "
        f"'{start}'. Run this notebook from within the project."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.analysis.eda import (
    compute_day_of_week_pattern,
    compute_rolling_stats,
    dataset_overview,
    plot_adi_cv2_quadrants,
    plot_representative_series,
    summarize_demand_classes,
    summarize_missing_values,
)
from src.data.validator import check_date_continuity, check_duplicate_keys
from src.utils.config import load_config

%matplotlib inline
pd.set_option("display.max_columns", None)

In [ ]:
config = load_config()
interim_dir = PROJECT_ROOT / config["data"]["interim_dir"]
processed_dir = PROJECT_ROOT / config["data"]["processed_dir"]
reports_dir = PROJECT_ROOT / config["reports"]["phase1_dir"]
reports_dir.mkdir(parents=True, exist_ok=True)

panel_path = interim_dir / config["output"]["panel_filename"]
classification_path = processed_dir / config["output"]["classification_filename"]
selected_path = processed_dir / config["output"]["selected_items_filename"]
aggregate_path = processed_dir / config["output"]["aggregate_filename"]

for p in [panel_path, classification_path, selected_path, aggregate_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"'{p}' not found. Run `python -m src.data.preprocessor` and "
            "`python -m src.analysis.demand_classification` first."
        )

panel = pd.read_parquet(panel_path)
classification = pd.read_parquet(classification_path)
selected = pd.read_parquet(selected_path)
aggregate = pd.read_parquet(aggregate_path)

adi_threshold = config["classification"]["adi_threshold"]
cv2_threshold = config["classification"]["cv2_threshold"]

print(f"Loaded panel ({panel.shape}), classification ({len(classification)} items), "
      f"selected ({len(selected)} items), aggregate ({len(aggregate)} days)")
print(f"Figures will be saved to: {reports_dir}")

## 1. Dataset overview

In [ ]:
overview = dataset_overview(panel)

print(f"Rows: {overview['n_rows']:,}")
print(f"Unique items: {overview['n_items']}")
print(f"Date range: {overview['date_min'].date()} to {overview['date_max'].date()} "
      f"({overview['n_days']} days)")
print(f"Inferred frequency: {overview['inferred_freq']}")
print(f"Store(s) in scope: {overview['stores']}")
print(f"Department(s) in scope: {overview['departments']}")
print()
print("Sales summary statistics:")
pd.Series(overview["sales_summary"])

## 2. Time-series continuity

Reuses the same `check_duplicate_keys` / `check_date_continuity`
functions the pipeline itself runs — this is checking the real rule
against the real data, not a separate EDA-only reimplementation of it.

In [ ]:
dup_result = check_duplicate_keys(panel)
cont_result = check_date_continuity(panel)
inferred_freq_on_aggregate = pd.infer_freq(aggregate["date"])

print(f"Duplicate item/date rows: {'none found' if dup_result.passed else dup_result.message}")
print(f"Date continuity: {'no gaps' if cont_result.passed else cont_result.message}")
print(f"Inferred frequency on the aggregate series: {inferred_freq_on_aggregate}")

## 3. Missing-value analysis

`is_expected=True` marks columns where missingness is a documented,
legitimate data characteristic (an event that didn't happen that day,
a price before an item was listed for sale — see
`src/data/preprocessor.py`), not a data-quality problem. Anything
`is_expected=False` here is worth a closer look.

In [ ]:
missing_summary = summarize_missing_values(panel)
missing_summary

In [ ]:
unexpected = missing_summary[~missing_summary["is_expected"]]
if unexpected.empty:
    print("No missing values outside the expected columns -- no data-quality flag raised.")
else:
    print("Missing values found outside the expected columns -- worth investigating:")
    print(unexpected)

fig, ax = plt.subplots(figsize=(7, 4))
colors = missing_summary["is_expected"].map({True: "#8C8C8C", False: "#8B3A3A"})
ax.barh(missing_summary["column"], missing_summary["n_missing"], color=colors)
ax.set_xlabel("Missing count")
ax.set_title("Missing values by column (gray = expected, red = unexpected)")
plt.tight_layout()
fig.savefig(reports_dir / "missing_values.png", dpi=150)
plt.show()

## 4. Aggregate demand analysis

Full CA_1 / FOODS_3 catalog summed into one daily series — descriptive
views only, nothing here is a model fit or a forecast.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(aggregate["date"], aggregate["sales"], color="#4C4C4C", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Total units sold")
ax.set_title("Aggregate daily demand — CA_1 / FOODS_3")
plt.tight_layout()
fig.savefig(reports_dir / "aggregate_sales.png", dpi=150)
plt.show()

In [ ]:
# Rolling mean/std: a smoothing view for visual inspection only, not a model.
ROLLING_WINDOW = 7  # one week
rolling = compute_rolling_stats(aggregate.set_index("date")["sales"], window=ROLLING_WINDOW)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(aggregate["date"], aggregate["sales"], color="#C9C9C9", linewidth=1, label="daily")
ax.plot(aggregate["date"], rolling["rolling_mean"].values, color="#4C4C4C", linewidth=1.5,
        label=f"{ROLLING_WINDOW}-day rolling mean")
ax.fill_between(
    aggregate["date"],
    rolling["rolling_mean"].values - rolling["rolling_std"].values,
    rolling["rolling_mean"].values + rolling["rolling_std"].values,
    color="#4C4C4C", alpha=0.15, label=f"{ROLLING_WINDOW}-day rolling std band",
)
ax.set_xlabel("Date")
ax.set_ylabel("Total units sold")
ax.set_title(f"Aggregate demand with {ROLLING_WINDOW}-day rolling mean/std")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
aggregate["sales"].hist(bins=20, ax=ax, color="#4C4C4C")
ax.set_xlabel("Total daily units sold (aggregate)")
ax.set_ylabel("Number of days")
ax.set_title("Distribution of aggregate daily demand")
plt.tight_layout()
plt.show()

In [ ]:
dow_pattern = compute_day_of_week_pattern(aggregate)

fig, ax = plt.subplots(figsize=(7, 4))
dow_pattern.plot(kind="bar", ax=ax, color="#4C4C4C")
ax.set_xlabel("Day of week")
ax.set_ylabel("Mean aggregate daily sales")
ax.set_title("Aggregate demand by day of week")
plt.tight_layout()
fig.savefig(reports_dir / "day_of_week_pattern.png", dpi=150)
plt.show()

## 5. Demand pattern analysis

From `data/processed/demand_classification.parquet` (the full CA_1 /
FOODS_3 catalog). `NoDemand` / `InsufficientData` are shown separately
if present, not folded into or excluded from the table.

In [ ]:
demand_summary = summarize_demand_classes(classification).set_index("demand_class")
demand_summary

In [ ]:
order = [c for c in ["Smooth", "Intermittent", "Erratic", "Lumpy",
                      "NoDemand", "InsufficientData"] if c in demand_summary.index]
colors = {"Smooth": "#4C4C4C", "Intermittent": "#8C8C8C", "Erratic": "#B0862B",
          "Lumpy": "#8B3A3A", "NoDemand": "#C9C9C9", "InsufficientData": "#C9C9C9"}

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(order, demand_summary.loc[order, "count"], color=[colors[c] for c in order])
ax.set_xlabel("Demand class")
ax.set_ylabel("Number of items")
ax.set_title("Demand category distribution — CA_1 / FOODS_3")
plt.tight_layout()
fig.savefig(reports_dir / "demand_category_distribution.png", dpi=150)
plt.show()

## 6. ADI vs CV² quadrants

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plot_adi_cv2_quadrants(classification, adi_threshold, cv2_threshold, ax=ax)
plt.tight_layout()
fig.savefig(reports_dir / "adi_cv2_quadrants.png", dpi=150)
plt.show()

## 7. Representative product visualization

Raw daily demand for the selected items, a couple per category — the
point is to see the shape difference directly: dense and steady
(Smooth) vs. sparse-but-regular (Intermittent) vs. frequent-but-spiky
(Erratic) vs. rare-and-spiky (Lumpy).

In [ ]:
fig = plot_representative_series(panel, selected, items_per_category=2)
fig.savefig(reports_dir / "representative_demand_patterns.png", dpi=150)
plt.show()

## 8. Temporal patterns

Each cell below computes something directly from the loaded data and
prints the actual result — nothing here is an assumed or pre-written
finding. Read the printed numbers against the plots above.

In [ ]:
# Trend: compare the first half of the observed window to the second half.
# Deliberately NOT a fitted regression line -- a period-over-period mean
# comparison stays clearly on the exploratory side, not the modeling side.
midpoint = len(aggregate) // 2
first_half_mean = aggregate["sales"].iloc[:midpoint].mean()
second_half_mean = aggregate["sales"].iloc[midpoint:].mean()
pct_change = (second_half_mean - first_half_mean) / first_half_mean * 100

print(f"First-half mean daily demand:  {first_half_mean:.1f}")
print(f"Second-half mean daily demand: {second_half_mean:.1f}")
print(f"Change: {pct_change:+.1f}%")
print("(a rough directional signal only -- not a trend model, and not a forecast)")

In [ ]:
# Highest/lowest demand days, cross-referenced against calendar events
# already joined into `panel` -- shows whether spikes/dips line up with
# a named event, without asserting that they do.
events_by_date = panel[["date", "event_name_1", "event_type_1",
                         "event_name_2", "event_type_2"]].drop_duplicates("date")

top5 = aggregate.nlargest(5, "sales").merge(events_by_date, on="date", how="left")
bottom5 = aggregate.nsmallest(5, "sales").merge(events_by_date, on="date", how="left")

print("Top 5 highest-demand days:")
print(top5[["date", "sales", "event_name_1", "event_name_2"]].to_string(index=False))
print()
print("Top 5 lowest-demand days:")
print(bottom5[["date", "sales", "event_name_1", "event_name_2"]].to_string(index=False))

In [ ]:
# Zero-demand behavior, and whether it shifts over the observed window.
midpoint_date = panel["date"].median()
zero_rate_first = (panel.loc[panel["date"] < midpoint_date, "sales"] == 0).mean() * 100
zero_rate_second = (panel.loc[panel["date"] >= midpoint_date, "sales"] == 0).mean() * 100

print(f"Zero-sales days, first half of the window:  {zero_rate_first:.1f}% of (item, day) rows")
print(f"Zero-sales days, second half of the window: {zero_rate_second:.1f}% of (item, day) rows")
print("Weekly seasonality: see the day-of-week bar chart in section 4.")

## Summary

Six figures saved to `reports/phase1/`: `aggregate_sales.png`,
`missing_values.png`, `day_of_week_pattern.png`,
`demand_category_distribution.png`, `adi_cv2_quadrants.png`,
`representative_demand_patterns.png`.

No decomposition was used (kept this notebook clearly on the
exploratory side, not modeling-adjacent), and no forecasting model,
fit, or metric appears anywhere above.